In [ ]:
%pip install --quiet opencv-python pillow pytesseract numpy pandas textblob


In [1]:
import os, re, json, cv2, numpy as np, pandas as pd
from PIL import Image
import pytesseract

# Optional: set Tesseract path on Windows if not on PATH
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

def _tesseract_text(img_bgr) -> str:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    thr  = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv2.THRESH_BINARY, 31, 11)
    return pytesseract.image_to_string(thr, config="--oem 3 --psm 6 -l eng")

def find_ingredient_section(pil_img: Image.Image) -> Image.Image:
    cv_image = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(cv_image, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w * h < 500:
            continue
        roi = cv_image[y:y+h, x:x+w]
        text = pytesseract.image_to_string(roi, config="--oem 3 --psm 6 -l eng")
        if "INGREDIENT" in text.upper():
            extend = int(0.6 * h) + 300
            y2 = min(cv_image.shape[0], y + h + extend)
            x2 = min(cv_image.shape[1], x + w + 50)
            return Image.fromarray(cv_image[..., ::-1]).crop((x, y, x2, y2))  # convert back to RGB PIL
    return Image.fromarray(cv_image[..., ::-1])

def extract_ingredients(text: str) -> list[str]:
    text_norm = re.sub(r"\s+", " ", text).strip()
    m = re.search(r"ingredients?\s*[:\-]?\s*(.+)", text_norm, flags=re.IGNORECASE)
    if m:
        candidates = m.group(1)
    else:
        m2 = re.search(r"([A-Za-z0-9\s\(\)\[\]\-]+(?:,\s*[A-Za-z0-9\s\(\)\[\]\-]+)+)", text_norm)
        candidates = m2.group(1) if m2 else ""
    if not candidates:
        return []
    parts = [re.sub(r"^[\s\.\-:;]+|[\s\.\-:;]+$", "", x) for x in candidates.split(",")]
    return [p for p in parts if p and len(p) > 1]

def process_image_file(path: str) -> dict:
    pil_img = Image.open(path).convert("RGB")
    cropped = find_ingredient_section(pil_img)
    cv_img  = cv2.cvtColor(np.array(cropped), cv2.COLOR_RGB2BGR)
    raw     = _tesseract_text(cv_img)
    return {"ingredients": extract_ingredients(raw), "raw_text": raw.strip()}


In [4]:
# === Batch process all images in a folder ===
import os, glob, json, pandas as pd

# 1) Set your folder path (use a raw string r"...")
DIR_PATH = r"C:\Users\guest441\Downloads\Lishebora_Version_2\Lishebora_Version_2\Ingredients"

# 2) Collect image files (recursively). Add/remove extensions if you like.
EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")
all_paths = [p for p in glob.glob(os.path.join(DIR_PATH, "**", "*.*"), recursive=True)
             if os.path.splitext(p)[1].lower() in EXTS]

if not all_paths:
    raise SystemExit(f"No images found under: {DIR_PATH}")

print(f"Found {len(all_paths)} image(s). Processing...")

rows = []
errors = []

for i, path in enumerate(all_paths, 1):
    try:
        result = process_image_file(path)  # uses the function you already defined in the notebook
        rows.append({
            "file": path,
            "n_ingredients": len(result["ingredients"]),
            "ingredients": "; ".join(result["ingredients"]),
            "raw_text": result["raw_text"]
        })
        if i % 5 == 0 or i == len(all_paths):
            print(f"  {i}/{len(all_paths)} done")
    except Exception as e:
        errors.append({"file": path, "error": str(e)})

# 3) Save a CSV summary next to the folder
out_csv = os.path.join(DIR_PATH, "ingredients_extracted_all.csv")
pd.DataFrame(rows).to_csv(out_csv, index=False, encoding="utf-8-sig")
print("Saved CSV:", out_csv)

# (Optional) Save a JSONL with full per-file details
out_jsonl = os.path.join(DIR_PATH, "ingredients_extracted_all.jsonl")
with open(out_jsonl, "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("Saved JSONL:", out_jsonl)

# (Optional) Show a quick preview (top 5)
pd.DataFrame(rows).head()


Found 182 image(s). Processing...
  5/182 done
  10/182 done
  15/182 done
  20/182 done
  25/182 done
  30/182 done
  35/182 done
  40/182 done
  45/182 done
  50/182 done
  55/182 done
  60/182 done
  65/182 done
  70/182 done
  75/182 done
  80/182 done
  85/182 done
  90/182 done
  95/182 done
  100/182 done
  105/182 done
  110/182 done
  115/182 done
  120/182 done
  125/182 done
  130/182 done
  135/182 done
  140/182 done
  145/182 done
  150/182 done
  155/182 done
  160/182 done
  165/182 done
  170/182 done
  175/182 done
  180/182 done
  182/182 done
Saved CSV: C:\Users\guest441\Downloads\Lishebora_Version_2\Lishebora_Version_2\Ingredients\ingredients_extracted_all.csv
Saved JSONL: C:\Users\guest441\Downloads\Lishebora_Version_2\Lishebora_Version_2\Ingredients\ingredients_extracted_all.jsonl


,file,n_ingredients,ingredients,raw_text
0,C:\Users\guest441\Downloads\Lishebora_Version_...,28,? CT] Potato; Refined Palmolein Oil; Bengat gr...,". INGREDIENTS: ?\nCT] Potato, Refined Palmolei..."
1,C:\Users\guest441\Downloads\Lishebora_Version_...,4,Torncto pures (88%); Salt; Famncrind concerti ...,"SINGREDIENTS; Torncto pures (88%), Salt, Famnc..."
2,C:\Users\guest441\Downloads\Lishebora_Version_...,2,Cake Concentrate [Milk Solids; starclia,"ingredients: Cake Concentrate [Milk Solids, st..."
3,C:\Users\guest441\Downloads\Lishebora_Version_...,0,,"ICardsinom Cheama coe pera Gn a Nimes"" rears\n..."
4,C:\Users\guest441\Downloads\Lishebora_Version_...,2,Eggs Ghee; Veg Shoreningr Honey,"faidar Sugar: Eggs Ghee, Veg Shoreningr Honey;..."
